In [1]:

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Audio
import lightning as L
import sys
from lightning.pytorch.callbacks import LearningRateMonitor

sys.path.append('../')
sys.path.append('./')
import importlib
import yaml
import torch

from tqdm.auto import tqdm
import logging

torch.set_float32_matmul_precision('medium')
# logging.getLogger('sox').setLevel(logging.ERROR)
# logger = logging.getLogger('sox')
# logger.setLevel('CRITICAL')


In [2]:
import lightning_scripts.lightning_ssl_matched_speech_in_noise as lightning 
importlib.reload(lightning)


LitAudioSSL = lightning.LitAudioSSL

## init config. Will be yaml eventually, but start as dict 
config_path = "model_configs/kell2018_sym_barlow_lmbda_1e-2_lr_2e-1_eq_lmbda_5e-01.yaml"
config = yaml.load(open(config_path, 'r'), Loader=yaml.FullLoader)

config['num_workers'] = 10
config['hparas']['batch_size'] = 64
# config['hparas']['ssl_loss'] = 'Symmetric_Paired_Loss'
config['hparas']['global_batch_size'] = 64
config['num_gpus'] = 1 


model = LitAudioSSL(config).cuda()



In [3]:
model.model

ModelWithFrontEnd(
  (front_end): AudioToAudioRepresentation(
    (rep): AudioToCochleagram(
      (envelope_extraction): HilbertEnvelopeExtraction()
      (downsampling_op): SincWithKaiserWindow()
      (Cochleagram): Cochleagram(
        (compute_subbands): ComputeSubbands()
        (envelope_extraction): HilbertEnvelopeExtraction()
        (downsampling): SincWithKaiserWindow()
      )
    )
    (compression): ClippedGradPower(
      (compression_function): ClippedGradPowerCompression()
    )
  )
  (model): SSLBaseModelDualTask(
    (f): AuditoryCNNMultiTask(
      (batchnorm0): BatchNorm2d(1, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv0): Conv2dSame(1, 96, kernel_size=(7, 14), stride=(3, 3), bias=False)
      (relu0): ReLU()
      (maxpool0): MaxPool2dSame(kernel_size=[2, 5], stride=[2, 2], padding=(0, 0), dilation=1, ceil_mode=True)
      (batchnorm1): BatchNorm2d(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv1): Con

In [3]:
## Get training batch

train_dataloader = model.train_dataloader()

In [5]:
logging.getLogger('sox').setLevel(logging.ERROR)
batch = next(iter(train_dataloader))
[spec_11, spec_12, spec_21, spec_22], [labels_11, labels_12, labels_21, labels_22] = batch


In [8]:
_, out_11, logits_11 = model.model(spec_11.cuda())
_, out_12, logits_12 = model.model(spec_12.cuda())
_, out_21, logits_21 = model.model(spec_21.cuda())
_, out_22, logits_22 = model.model(spec_22.cuda())

In [21]:
from importlib import reload
from lightning_scripts.losses import sym_paired_loss
reload(sym_paired_loss)

loss_kwargs = config['hparas'].get('ssl_loss_kwargs', None) 

loss_fn_inv_kwargs = loss_kwargs.get('loss_fn_inv_kwargs', None) 
loss_fn_eq_kwargs = loss_kwargs.get('loss_fn_eq_kwargs', None) 
loss_fn_inv = model.get_loss_fn(loss_kwargs['loss_fn_inv'], loss_fn_inv_kwargs)
loss_fn_eq = model.get_loss_fn(loss_kwargs['loss_fn_eq'], loss_fn_eq_kwargs)


loss_fn = sym_paired_loss.Symmetric_Paired_Loss(loss_fn_inv=loss_fn_inv, loss_fn_eq=loss_fn_eq, lmda=loss_kwargs['lmda']).cuda()
loss_fn(out_11, out_12, out_21, out_22)

(tensor(12.2347, device='cuda:0', grad_fn=<AddBackward0>),
 tensor(13.4005, device='cuda:0', grad_fn=<MulBackward0>),
 tensor(11.0688, device='cuda:0', grad_fn=<MulBackward0>))

In [4]:
trainer = L.Trainer(
                    # callbacks=[lr_monitor],
                    # limit_train_batches=5,
                    limit_val_batches=2,
                    max_epochs=5,
                    callbacks=None,
                    #  strategy='ddp_notebook',
                    #  reload_dataloaders_every_n_epochs=-1,
                    devices=1)
logging.getLogger('sox').setLevel(logging.ERROR)

trainer.fit(model)

/mnt/home/igriffith/envs/cochdnn_ssl_pl/lib/python3.12/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /mnt/home/igriffith/envs/cochdnn_ssl_pl/lib/python3. ...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/mnt/home/igriffith/envs/cochdnn_ssl_pl/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(

  | Name            | Type                       | Params | Mode 
-----------------------------------------------------------------------
0 | audio_rep       | AudioToAudioRepresentation | 0      | train
1 | model           | ModelWithFrontEnd          | 246 M  

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Rank 0 N training batches 317


Training: |          | 0/? [00:00<?, ?it/s]


Detected KeyboardInterrupt, attempting graceful shutdown ...


NameError: name 'exit' is not defined